# Generalization: irrelevant context length in SQuAD 1.1

This notebook evaluates the **published seed-17 epoch-2 LoRA adapter** against its frozen Qwen3-0.6B base model. It asks whether adding irrelevant text **before the answer-bearing context** reduces extractive QA quality. It uses the exact 900-question SQuAD 1.1 test set in `published/seed17_epoch2/data/squad_test.jsonl`, verified against `reference/sample_ids.json`, and selects two questions per article for the stress test. These articles were held out from LoRA fine-tuning. Original-condition scores in this subset should not be presented as the original 900-question aggregate.

**Predeclared hypothesis and quantitative prediction (before running):** At roughly 1,024 additional context tokens, the LoRA model's exact match (EM) will be at least 8 percentage points lower than on the same unmodified questions. We expect LoRA to remain ahead of the base model. The 512-token condition checks the shape of the effect.

**Control:** same questions, gold answers, base weights, adapter, prompt, greedy decoding, and 64-token output cap. Only the prepended irrelevant text changes. Two fixed distractor seeds repeat the comparison. We report exact match, token F1, per-seed results, article-cluster bootstrap intervals, and failure cases. Distractor content changes along with length, so this experiment measures robustness to *extra irrelevant context*, not an isolated positional effect. One trained adapter and one dataset remain limitations.

Run from top to bottom in **Google Colab with a GPU**. The notebook clones the public GitHub repository, verifies the publication bundle, and downloads the pinned Qwen3-0.6B base model on first use. It writes CSV, JSON, and a compact report to `OUTPUT_DIR` in Drive. No training is performed.


In [ ]:
# 1. Setup. Colab's installed torch is retained; install the training notebook's library versions.
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # Must precede importing torch.
%pip -q install "transformers==4.53.3" "tokenizers==0.21.4" "peft==0.17.1" "accelerate==1.8.1" "huggingface_hub==0.36.2" "numpy==2.2.6" "safetensors==0.8.0" "pandas==2.2.3" "matplotlib==3.10.0"

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import subprocess
ROOT = Path('/content/LLM-project')
PINNED_COMMIT = '2359fd5d9ea34bb9d1845fe1a6732d657343361f'
if not ROOT.exists():
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/sunweiyi2026-ship-it/LLM-project.git', str(ROOT)], check=True)
subprocess.run(['git', '-C', str(ROOT), 'fetch', '--depth', '1', 'origin', PINNED_COMMIT], check=True)
subprocess.run(['git', '-C', str(ROOT), 'checkout', '--detach', PINNED_COMMIT], check=True)
REPO_COMMIT = subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], text=True).strip()
assert REPO_COMMIT == PINNED_COMMIT
OUTPUT_DIR = Path('/content/drive/MyDrive/LLM-project/generalization_length_published_seed17_epoch2')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert (ROOT / 'published/seed17_epoch2/manifest.json').is_file()
print('Published model and data:', ROOT / 'published/seed17_epoch2', '| commit:', REPO_COMMIT)
print('Outputs:', OUTPUT_DIR)


In [ ]:
# 2. Verify the published 900-question list and load the pinned model + adapter.
import hashlib, json, sys
from collections import defaultdict
from contextlib import nullcontext
import numpy as np
import pandas as pd
import torch
sys.path.insert(0, str(ROOT / 'src'))
from genqa.pretrained import verify_published, load_published_model
from genqa import engine

assert torch.cuda.is_available(), 'Choose a GPU runtime in Colab.'
verification = verify_published(ROOT)
loaded = load_published_model(ROOT)
publication = loaded['spec']
test_rows = loaded['rows']['test']
original_ids = json.loads((ROOT / 'published/seed17_epoch2/reference/sample_ids.json').read_text())['test']
assert len(test_rows) == len(original_ids) == 900
assert [r['id'] for r in test_rows] == original_ids, 'Test IDs/order differ from original run'
assert len({r['article_title'] for r in test_rows}) == 45
model, tokenizer = loaded['model'], loaded['tokenizer']
MODEL_ID = publication['model']['model_id']
REVISION = publication['model']['revision']
PROMPT = loaded['prompt']['instruction']
MAX_NEW_TOKENS = publication['evaluation']['max_new_tokens']
BATCH_SIZE = publication['evaluation']['batch_size']
SAMPLE_SEED = 731
DISTRACTOR_SEEDS = (101, 202)
TARGET_EXTRA_TOKENS = (512, 1024)
print('Verified:', verification['status'], '| 900 IDs:', len(original_ids),
      '| GPU:', torch.cuda.get_device_name(0), '| source run:', publication['source_run']['name'])
reference_metrics = pd.read_csv(ROOT / 'published/seed17_epoch2/reference/metrics.csv')
display(reference_metrics[reference_metrics['phase'].eq('test')][['method', 'phase', 'n', 'EM (%)', 'F1 (%)']])


In [ ]:
# 3. Deterministic subset: two question IDs per original test article.
# All samples and distractors are fixed before evaluation. Distractors come from other
# test articles, not the training set, and the current gold answer is filtered out.
by_article = defaultdict(list)
paragraphs = defaultdict(list)
for row in test_rows:
    title = row['article_title']
    paragraphs[title].append(row['context'])
    by_article[title].append({**row, 'article': title})

def stable_key(value):
    return hashlib.sha256(f'{SAMPLE_SEED}:{value}'.encode()).hexdigest()

selected = []
for title in sorted(by_article):
    selected.extend(sorted(by_article[title], key=lambda r: stable_key(r['id']))[:2])
assert len(selected) == 2 * len(by_article)
assert len({r['id'] for r in selected}) == len(selected)
pool = [(title, context) for title in sorted(paragraphs)
        for context in sorted(set(paragraphs[title]))]
print('Questions:', len(selected), '| articles:', len(by_article))

def context_tokens(text):
    return tokenizer.encode(text, add_special_tokens=False)

def distractor(row, target, seed):
    # Same shuffled pool for 512 and 1024, so the shorter condition is a prefix of the longer.
    key = int(hashlib.sha256(f'{seed}:{row["id"]}'.encode()).hexdigest()[:16], 16)
    rng = np.random.default_rng(key)
    order = rng.permutation(len(pool))
    tokens = []
    answer_texts = [a.lower().strip() for a in row['answers'] if len(a.strip()) >= 4]
    for index in order:
        title, text = pool[int(index)]
        if title == row['article'] or text == row['context']:
            continue
        if any(a in text.lower() for a in answer_texts):
            continue
        tokens.extend(context_tokens(text + '\n\n'))
        if len(tokens) >= target:
            return tokenizer.decode(tokens[:target], skip_special_tokens=True).strip()
    raise RuntimeError('Insufficient distractor text')

conditions = [('original', 0, None)] + [
    (f'extra_{size}_seed_{seed}', size, seed)
    for size in TARGET_EXTRA_TOKENS for seed in DISTRACTOR_SEEDS
]
cases = []
for row in selected:
    for condition, extra, seed in conditions:
        added = distractor(row, extra, seed) if extra else ''
        context = f'{added}\n\n{row["context"]}' if added else row['context']
        cases.append({**row, 'condition': condition, 'extra_target_tokens': extra,
                      'distractor_seed': seed, 'eval_context': context})
print('Total input cases per model:', len(cases))

for row in cases:
    prepared = engine.prepare_input({**row, 'context': row['eval_context']}, tokenizer,
                                    loaded['prompt'], publication['evaluation']['max_input_tokens'])
    assert not prepared['context_truncated'], f'Input was truncated: {row["id"]}'
    row['prompt_tokens'] = prepared['input_tokens']

manifest = {
    'question': 'How much does irrelevant prepended context hurt SQuAD QA?',
    'hypothesis': 'LoRA EM falls by >=8 percentage points at +1024 tokens vs original',
    'publication': 'published/seed17_epoch2/manifest.json',
    'repository_commit': REPO_COMMIT,
    'publication_verification': verification,
    'test_rows_sha256': publication['data']['test']['rows_sha256'],
    'model_id': MODEL_ID, 'revision': REVISION,
    'adapter_sha256': publication['files']['adapter/adapter_model.safetensors']['sha256'],
    'source_run': publication['source_run'], 'sample_seed': SAMPLE_SEED,
    'distractor_seeds': DISTRACTOR_SEEDS, 'target_extra_tokens': TARGET_EXTRA_TOKENS,
    'article_count': len(by_article), 'question_ids': [r['id'] for r in selected],
    'prompt': PROMPT, 'generation': {'do_sample': False, 'max_new_tokens': MAX_NEW_TOKENS,
                                   'enable_thinking': False, 'batch_size': BATCH_SIZE},
    'max_prompt_tokens': max(r['prompt_tokens'] for r in cases)
}
manifest_path = OUTPUT_DIR / 'manifest.json'
if manifest_path.exists():
    old = json.loads(manifest_path.read_text())
    assert old == manifest, 'Output directory belongs to a different experiment; choose a new OUTPUT_DIR'
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False))


In [ ]:
# 4. Evaluate with the repository's original generation and scoring engine.
# This keeps prompt tokenization, EOS handling, greedy generation, and SQuAD scoring
# aligned with the published run. Completed conditions are reused after a disconnect.
def result_path(method, condition):
    return OUTPUT_DIR / method.lower() / condition / 'predictions.jsonl'

for method in ('Baseline', 'LoRA'):
    for condition, extra, seed in conditions:
        rows = [{**r, 'context': r['eval_context']} for r in cases if r['condition'] == condition]
        destination = result_path(method, condition).parent
        if destination.exists():
            metrics_path = destination / 'metrics.json'
            assert metrics_path.is_file(), f'Incomplete condition directory: {destination}'
            metrics = json.loads(metrics_path.read_text())
            assert metrics['complete'] and metrics['n_predictions'] == len(rows)
            completed = [json.loads(line) for line in result_path(method, condition).read_text().splitlines() if line]
            assert {r['id'] for r in completed} == {r['id'] for r in rows}
            print('Reused complete condition:', method, condition)
            continue
        print('Evaluating:', method, condition, len(rows), 'questions', flush=True)
        adapter_context = model.disable_adapter() if method == 'Baseline' else nullcontext()
        with adapter_context:
            engine.evaluate(model, tokenizer, rows, loaded['prompt'], destination,
                            batch_size=BATCH_SIZE,
                            max_input_tokens=publication['evaluation']['max_input_tokens'],
                            max_new_tokens=MAX_NEW_TOKENS, loss_examples_per_split=0,
                            seed=publication['evaluation']['seed'],
                            run_label=f'Generalization {method} {condition}')


In [ ]:
# 5. Paired analysis, article-cluster bootstrap intervals, and failure cases.
records = []
for method in ('Baseline', 'LoRA'):
    for condition, _, _ in conditions:
        with result_path(method, condition).open() as f:
            for line in f:
                if line.strip():
                    row = json.loads(line)
                    score = engine.score_answer(row['prediction'], row['answers'])
                    records.append({**row, 'method': method, 'em': score['em']/100,
                                    'f1': score['f1']/100, 'prompt_tokens': row['input_tokens']})
pred = pd.DataFrame(records)
assert not pred.duplicated(['method', 'condition', 'id']).any()
assert len(pred) == 2 * len(conditions) * len(selected)
pred.to_csv(OUTPUT_DIR / 'all_predictions.csv', index=False)

summary = (pred.groupby(['method', 'extra_target_tokens', 'distractor_seed'], dropna=False)
           .agg(n=('id', 'count'), articles=('article', 'nunique'),
                em=('em', 'mean'), f1=('f1', 'mean'),
                mean_prompt_tokens=('prompt_tokens', 'mean')).reset_index())
summary[['em', 'f1']] *= 100
summary.to_csv(OUTPUT_DIR / 'condition_summary.csv', index=False)
display(summary.round(2))

def paired_article_effect(method, extra, metric, bootstrap=2000):
    subset = pred[pred.method.eq(method)]
    original = subset[subset.extra_target_tokens.eq(0)].set_index('id')[metric]
    changed = (subset[subset.extra_target_tokens.eq(extra)]
               .groupby(['article', 'id'])[metric].mean().reset_index())
    changed['difference'] = changed[metric] - changed['id'].map(original)
    article_values = changed.groupby('article')['difference'].mean().to_numpy()
    rng = np.random.default_rng(42)
    draws = rng.choice(article_values, size=(bootstrap, len(article_values)), replace=True).mean(axis=1)
    return 100 * article_values.mean(), 100 * np.quantile(draws, [0.025, 0.975])

effects = []
for method in ('Baseline', 'LoRA'):
    for extra in TARGET_EXTRA_TOKENS:
        for metric in ('em', 'f1'):
            estimate, interval = paired_article_effect(method, extra, metric)
            effects.append({'method': method, 'extra_target_tokens': extra, 'metric': metric,
                            'change_pp': estimate, 'ci95_low_pp': interval[0],
                            'ci95_high_pp': interval[1]})
effects = pd.DataFrame(effects)
effects.to_csv(OUTPUT_DIR / 'paired_effects.csv', index=False)
display(effects.round(2))

base = pred[(pred.method.eq('LoRA')) & (pred.extra_target_tokens.eq(0))]
stressed = pred[(pred.method.eq('LoRA')) & (pred.extra_target_tokens.eq(1024))]
wide = stressed.merge(base[['id', 'em', 'prediction']].rename(
    columns={'em': 'base_em', 'prediction': 'base_prediction'}), on='id', validate='many_to_one')
failures = wide[(wide.base_em.eq(1)) & (wide.em.eq(0))].sort_values(
    ['distractor_seed', 'article', 'id'])
failures[['id', 'article', 'question', 'answers', 'distractor_seed',
          'base_prediction', 'prediction']].to_csv(OUTPUT_DIR / 'lora_failures_1024.csv', index=False)
display(failures[['article', 'question', 'answers', 'distractor_seed',
                  'base_prediction', 'prediction']].head(12))

row = effects[(effects.method.eq('LoRA')) & (effects.extra_target_tokens.eq(1024)) &
              (effects.metric.eq('em'))].iloc[0]
prediction_met = bool(row.change_pp <= -8.0)
report = f'''# Generalization result: irrelevant preceding context

Predeclared prediction: LoRA EM falls by at least 8 percentage points at +1024 tokens.
Observed paired mean change: {row.change_pp:.2f} pp (95% article-cluster bootstrap interval {row.ci95_low_pp:.2f} to {row.ci95_high_pp:.2f} pp).
Point-estimate threshold met: {prediction_met}. Interpret this alongside the interval, not as a separate significance test.

Sample: {len(selected)} fixed questions from the published 900-question SQuAD 1.1 test set, two questions per article across {len(by_article)} articles held out from LoRA fine-tuning. Test IDs and order were checked against the published original `reference/sample_ids.json`. Each length condition uses two fixed distractor seeds. Generation is greedy with 64 new tokens at most. The same base model and epoch-2 adapter were used for all cases.

Interpretation: this measures robustness to additional irrelevant context placed before the answer-bearing passage on a subset of the published original test set. It does not isolate position from content or establish cross-domain transfer. The adapter has one training seed. The subset's original-condition scores should not be confused with the full 900-question aggregate.

See `condition_summary.csv`, `paired_effects.csv`, `all_predictions.csv`, and `lora_failures_1024.csv` for evidence and examples.
'''
(OUTPUT_DIR / 'report.md').write_text(report)
print(report)


## 5b. Direct comparison on existing predictions

This calculation uses the completed `all_predictions.csv`; **it runs no inference**. For each question, average the two +1024 distractor seeds, subtract that model's original EM, then subtract Baseline's change from LoRA's change for the same question. Bootstrap the 45 article means (two questions per article), keeping both model predictions paired. The 95% percentile interval is descriptive because this comparison was added after viewing the main results. Complete this step before the placement follow-up below.


In [ ]:
# 5b. Direct paired LoRA-minus-Baseline effect at +1024, before placement follow-up.
existing = pd.read_csv(OUTPUT_DIR / 'all_predictions.csv', dtype={'id': str})
assert len(existing) == 2 * len(conditions) * len(selected)

def article_changes(method, metric='em'):
    subset = existing[existing.method.eq(method)]
    original = subset[subset.extra_target_tokens.eq(0)].set_index('id')[metric]
    stressed = (subset[subset.extra_target_tokens.eq(1024)]
                .groupby(['article', 'id'])[metric].mean().reset_index())
    stressed['change'] = stressed[metric] - stressed['id'].map(original)
    assert stressed['change'].notna().all() and len(stressed) == len(selected)
    return stressed.set_index(['article', 'id'])['change']

interaction_rows = []
for metric in ('em', 'f1'):
    paired = article_changes('LoRA', metric) - article_changes('Baseline', metric)
    assert paired.notna().all() and len(paired) == len(selected)
    article_means = paired.groupby(level='article').mean().to_numpy()
    assert len(article_means) == len(by_article)
    rng = np.random.default_rng(42)
    draws = rng.choice(article_means, size=(2000, len(article_means)), replace=True).mean(axis=1)
    low, high = np.quantile(draws, [0.025, 0.975])
    interaction_rows.append({'metric': metric,
                             'difference_in_change_pp': 100 * article_means.mean(),
                             'ci95_low_pp': 100 * low, 'ci95_high_pp': 100 * high,
                             'questions': len(paired), 'articles': len(article_means),
                             'bootstrap_draws': len(draws)})
interaction = pd.DataFrame(interaction_rows)
interaction.to_csv(OUTPUT_DIR / 'exploratory_interaction.csv', index=False)
display(interaction.round(2))
em_interaction = interaction[interaction.metric.eq('em')].iloc[0]
(OUTPUT_DIR / 'relative_robustness_report.md').write_text(
    '# Direct paired comparison at +1024 tokens\n\n'
    f"LoRA minus Baseline change in EM: {em_interaction.difference_in_change_pp:+.2f} pp "
    f"(95% article-cluster bootstrap interval {em_interaction.ci95_low_pp:+.2f} to "
    f"{em_interaction.ci95_high_pp:+.2f} pp).\n\n"
    'This is an exploratory paired comparison on 90 fixed questions in 45 articles, '
    'averaging two distractor seeds per question. The interval crosses zero, so the '
    'data do not establish that LoRA is more robust than Baseline. No new inference was run.\n'
)
